# Feature Inspection
**Cohort overview + per-patient inspector**

Point `FEATURE_CSV` at merged feature file and run all cells.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────
FEATURE_CSV = Path('features_csv/all_features.csv')   # ← change if needed
# ──────────────────────────────────────────────────────────────────────

GROUP_COLS   = ['Control', 'iRBD', 'PD(-RBD)', 'PD(+RBD)', 'PLM']
GROUP_COLORS = {
    'Control' : '#378ADD',
    'iRBD'    : '#BA7517',
    'PD(-RBD)': '#639922',
    'PD(+RBD)': '#D85A30',
    'PLM'     : '#888780',
    'Unknown' : '#AAAAAA',
}
CAT_PREFIXES = [
    ('EOG amplitude',        r'^rem_(loc|roc)_'),
    ('EM classification',    r'^(sem_|rem_em_|em_count)'),
    ('Phasic / tonic',       r'^(phasic|tonic)'),
    ('Bout features',        r'^bout_'),
    ('GSSC / sleep staging', r'^(prob_|rem_stability|rem_fragmentation|sleep_efficiency|total_sleep|stage_frac|n_epochs)'),
    ('EEG band power',       r'^eeg__'),
    ('REM duration',         r'^rem_(duration|epoch)'),
]
SKIP_COLS = set(GROUP_COLS + ['subject_id', 'DCSM_ID', '_group'])

df_raw = pd.read_csv(FEATURE_CSV, low_memory=False)
print(f'Loaded {df_raw.shape[0]} subjects × {df_raw.shape[1]} columns')

def assign_group(row):
    for g in GROUP_COLS:
        if g in row.index and pd.notna(row[g]) and int(row[g]) == 1:
            return g
    return 'Unknown'

df = df_raw.copy()
df['_group'] = df.apply(assign_group, axis=1)

num_cols = [
    c for c in df.columns
    if c not in SKIP_COLS
    and pd.api.types.is_numeric_dtype(df[c])
    and not all(df[c].isna())
]
print(f'Numeric feature columns: {len(num_cols)}')
print(f'Groups: {df["_group"].value_counts().to_dict()}')

---
## 1  Cohort overview

In [ ]:
nan_per_subject = df[num_cols].isna().sum(axis=1)
n_total   = len(df)
avg_nan   = nan_per_subject.mean()
n_any_nan = (nan_per_subject > 0).sum()
n_complete= (nan_per_subject == 0).sum()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Cohort overview', fontsize=13, fontweight='bold', y=1.01)

# ── 1a: summary table ────────────────────────────────────────────────
ax = axes[0]
ax.axis('off')
summary = [
    ['Subjects',               str(n_total)],
    ['Feature columns',        str(len(num_cols))],
    ['Avg NaN / subject',      f'{avg_nan:.1f}'],
    ['Subjects with any NaN',  f'{n_any_nan}  ({n_any_nan/n_total*100:.0f}%)'],
    ['Complete subjects',       f'{n_complete}  ({n_complete/n_total*100:.0f}%)'],
]
tbl = ax.table(cellText=summary, colLabels=['Metric', 'Value'],
               cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#CCCCCC')
    if r == 0: cell.set_facecolor('#F0F0F0')
ax.set_title('Summary', fontsize=11, pad=8)

# ── 1b: group counts ─────────────────────────────────────────────────
ax = axes[1]
group_counts = df['_group'].value_counts()
groups = [g for g in list(GROUP_COLORS.keys()) if g in group_counts]
counts = [group_counts[g] for g in groups]
colors = [GROUP_COLORS[g] for g in groups]
bars = ax.barh(groups, counts, color=colors, edgecolor='white', height=0.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(cnt), va='center', fontsize=10)
ax.set_xlabel('N subjects', fontsize=10)
ax.set_title('Group breakdown', fontsize=11, pad=8)
ax.spines[['top','right']].set_visible(False)

# ── 1c: NaN per subject histogram ────────────────────────────────────
ax = axes[2]
ax.hist(nan_per_subject, bins=20, color='#378ADD', edgecolor='white', linewidth=0.5)
ax.axvline(avg_nan, color='#D85A30', linewidth=1.5, linestyle='--', label=f'mean = {avg_nan:.1f}')
ax.set_xlabel('NaN features per subject', fontsize=10)
ax.set_ylabel('N subjects', fontsize=10)
ax.set_title('NaN distribution across subjects', fontsize=11, pad=8)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

### 1.1  Feature category completeness

In [ ]:
cat_stats = []
for label, pattern in CAT_PREFIXES:
    cols = [c for c in num_cols if pd.Series([c]).str.match(pattern).any()]
    if not cols:
        continue
    nan_frac = df[cols].isna().mean().mean()
    cat_stats.append({'category': label, 'n_features': len(cols),
                      'completeness': (1 - nan_frac) * 100})

# features not matched by any prefix
matched = set(c for _, pat in CAT_PREFIXES
              for c in num_cols if pd.Series([c]).str.match(pat).any())
other = [c for c in num_cols if c not in matched]
if other:
    nan_frac = df[other].isna().mean().mean()
    cat_stats.append({'category': 'Other / uncategorised', 'n_features': len(other),
                      'completeness': (1 - nan_frac) * 100})

cat_df = pd.DataFrame(cat_stats).sort_values('completeness')

fig, ax = plt.subplots(figsize=(10, 0.6 * len(cat_df) + 1.5))
bar_colors = ['#D85A30' if r < 50 else '#BA7517' if r < 80 else '#639922'
              for r in cat_df['completeness']]
bars = ax.barh(cat_df['category'], cat_df['completeness'],
               color=bar_colors, edgecolor='white', height=0.5)
for bar, row in zip(bars, cat_df.itertuples()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{row.completeness:.0f}%  ({row.n_features} feats)",
            va='center', fontsize=9, color='#444')
ax.set_xlim(0, 115)
ax.set_xlabel('Completeness (%)', fontsize=10)
ax.set_title('Feature category completeness  (green ≥ 80%, amber ≥ 50%, red < 50%)',
             fontsize=11, pad=8)
ax.axvline(80, color='#639922', linewidth=0.8, linestyle='--', alpha=0.6)
ax.axvline(50, color='#BA7517', linewidth=0.8, linestyle='--', alpha=0.6)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

### 1.2  NaN audit — features ranked by missing rate

In [ ]:
# ── CONFIG ─────────────────────────────────────
NAN_THRESH_PCT = 5    # show features with >= this % NaN
GROUP_FILTER   = None # set to e.g. 'iRBD' to filter, or None for all
# ───────────────────────────────────────────────

subset = df[df['_group'] == GROUP_FILTER] if GROUP_FILTER else df

nan_rates = (subset[num_cols].isna().mean() * 100).sort_values(ascending=False)
nan_rates = nan_rates[nan_rates >= NAN_THRESH_PCT]

if nan_rates.empty:
    print(f'No features with NaN rate >= {NAN_THRESH_PCT}% ' +
          (f'in group {GROUP_FILTER}' if GROUP_FILTER else 'across all subjects'))
else:
    label = f'group: {GROUP_FILTER}' if GROUP_FILTER else 'all subjects'
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(nan_rates) + 1)))
    colors = ['#D85A30' if r >= 50 else '#BA7517' if r >= 20 else '#378ADD'
              for r in nan_rates]
    ax.barh(nan_rates.index[::-1], nan_rates.values[::-1],
            color=colors[::-1], edgecolor='white', height=0.6)
    ax.set_xlabel('NaN rate (%)', fontsize=10)
    ax.set_title(f'Features with NaN >= {NAN_THRESH_PCT}%  [{label}]  '
                 f'({len(nan_rates)} features)', fontsize=11, pad=8)
    ax.axvline(50, color='#D85A30', linewidth=0.8, linestyle='--', alpha=0.7, label='50%')
    ax.axvline(20, color='#BA7517', linewidth=0.8, linestyle='--', alpha=0.7, label='20%')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    for i, (feat, val) in enumerate(zip(nan_rates.index[::-1], nan_rates.values[::-1])):
        ax.text(val + 0.3, i, f'{val:.0f}%', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f'\n{len(nan_rates)} features shown')

### 1.3  Feature distributions by group

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
FEATURES_TO_PLOT = None   # None = all; or list e.g. ['phasic_fraction', 'rem_em_rate_per_min']
N_COLS           = 4      # columns per row
CATEGORY_FILTER  = None   # None = all; or e.g. 'Phasic / tonic'
# ───────────────────────────────────────────────────────────────────────

if FEATURES_TO_PLOT is not None:
    plot_feats = [f for f in FEATURES_TO_PLOT if f in num_cols]
elif CATEGORY_FILTER is not None:
    pat = dict(CAT_PREFIXES).get(CATEGORY_FILTER, '')
    plot_feats = [c for c in num_cols if pd.Series([c]).str.match(pat).any()] if pat else num_cols
else:
    plot_feats = num_cols

groups_present = [g for g in GROUP_COLORS if g in df['_group'].values]
n_feats = len(plot_feats)
n_rows  = int(np.ceil(n_feats / N_COLS))

print(f'Plotting {n_feats} features across {n_rows} rows...')

fig, axes = plt.subplots(n_rows, N_COLS,
                          figsize=(N_COLS * 3.5, n_rows * 3.2),
                          squeeze=False)

for idx, feat in enumerate(plot_feats):
    ax = axes[idx // N_COLS][idx % N_COLS]
    data_by_group = [df.loc[df['_group'] == g, feat].dropna().values
                     for g in groups_present]
    bp = ax.boxplot(data_by_group, patch_artist=True, widths=0.5,
                    medianprops=dict(color='white', linewidth=1.5),
                    whiskerprops=dict(linewidth=0.8),
                    capprops=dict(linewidth=0.8),
                    flierprops=dict(marker='.', markersize=3, alpha=0.4))
    for patch, g in zip(bp['boxes'], groups_present):
        patch.set_facecolor(GROUP_COLORS[g])
        patch.set_alpha(0.75)
    # jitter
    for xi, (g, vals) in enumerate(zip(groups_present, data_by_group), 1):
        jitter = np.random.uniform(-0.15, 0.15, len(vals))
        ax.scatter(xi + jitter, vals, s=8, alpha=0.5,
                   color=GROUP_COLORS[g], zorder=3, edgecolors='none')
    nan_counts = [df[df['_group'] == g][feat].isna().sum() for g in groups_present]
    labels = [f"{g}\n(n={len(d)}, {nc} NaN)"
              for g, d, nc in zip(groups_present, data_by_group, nan_counts)]
    ax.set_xticks(range(1, len(groups_present)+1))
    ax.set_xticklabels(labels, fontsize=6, rotation=20, ha='right')
    ax.set_title(feat, fontsize=8, fontweight='bold', pad=3)
    ax.tick_params(labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

for idx in range(n_feats, n_rows * N_COLS):
    axes[idx // N_COLS][idx % N_COLS].set_visible(False)

legend_patches = [mpatches.Patch(color=GROUP_COLORS[g], label=g, alpha=0.75)
                  for g in groups_present]
fig.legend(handles=legend_patches, loc='upper center',
           ncol=len(groups_present), fontsize=9,
           title='Diagnostic group', title_fontsize=9,
           bbox_to_anchor=(0.5, 1.0), frameon=True)
fig.suptitle('Feature distributions by diagnostic group', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2  Patient inspector

Set `PATIENT_ID` to any `subject_id` / `DCSM_ID` in the dataset.

In [ ]:
# ── List all available patient IDs ──────────────────────────────────
id_col = 'subject_id' if 'subject_id' in df.columns else 'DCSM_ID'
available = df[[id_col, '_group']].copy()
available.columns = ['patient_id', 'group']
nan_counts_all = df[num_cols].isna().sum(axis=1).values
available['n_nan'] = nan_counts_all
available['pct_nan'] = (nan_counts_all / len(num_cols) * 100).round(1)

print(f'Available patients ({len(available)} total):')
print(available.sort_values(['group','patient_id']).to_string(index=False))

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────
PATIENT_ID = 'DCSM_1_a'   # ← change to the patient you want to inspect
# ──────────────────────────────────────────────────────────────────────

id_col = 'subject_id' if 'subject_id' in df.columns else 'DCSM_ID'
match = df[df[id_col].astype(str).str.contains(PATIENT_ID, na=False)]
if match.empty:
    print(f'Patient "{PATIENT_ID}" not found. Check the list above.')
else:
    row = match.iloc[0]
    group = row['_group']
    n_nan = row[num_cols].isna().sum()
    nan_feats = [c for c in num_cols if pd.isna(row[c])]
    ok_feats  = [c for c in num_cols if not pd.isna(row[c])]

    print('=' * 60)
    print(f'  Patient   : {row[id_col]}')
    print(f'  Group     : {group}')
    print(f'  NaN feats : {n_nan} / {len(num_cols)} ({n_nan/len(num_cols)*100:.0f}%)')
    print('=' * 60)

    if nan_feats:
        print(f'\n  MISSING features ({len(nan_feats)}):')
        for f in nan_feats:
            print(f'    [NaN]  {f}')

    print(f'\n  OK features ({len(ok_feats)}):')
    for f in ok_feats:
        val = row[f]
        print(f'    {f:<55}  {val:.4g}')

In [ ]:
# ── Patient: features organised by category ──────────────────────────
# Runs after the cell above — PATIENT_ID must be set.

if not match.empty:
    row = match.iloc[0]
    n_cats = len(CAT_PREFIXES)
    fig, axes = plt.subplots(n_cats, 1,
                              figsize=(14, n_cats * 2.2),
                              squeeze=False)

    for ax_row, (label, pattern) in enumerate(CAT_PREFIXES):
        ax = axes[ax_row][0]
        cols = [c for c in num_cols if pd.Series([c]).str.match(pattern).any()]
        if not cols:
            ax.text(0.5, 0.5, f'No features matched for "{label}"',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=9, color='gray')
            ax.set_title(label, fontsize=9, fontweight='bold')
            ax.axis('off')
            continue

        vals    = [row[c] for c in cols]
        is_nan  = [pd.isna(v) for v in vals]
        display = [float(v) if not pd.isna(v) else 0.0 for v in vals]
        bar_colors = ['#D85A30' if n else '#378ADD' for n in is_nan]

        # cohort medians for context
        medians = [df[c].median() for c in cols]

        x = np.arange(len(cols))
        ax.bar(x - 0.2, display, width=0.35, color=bar_colors,
               alpha=0.8, label='Patient value')
        ax.bar(x + 0.2, medians, width=0.35, color='#AAAAAA',
               alpha=0.6, label='Cohort median')

        for xi, (v, nan_flag) in enumerate(zip(vals, is_nan)):
            if nan_flag:
                ax.text(xi - 0.2, 0, 'NaN', ha='center', va='bottom',
                        fontsize=7, color='#D85A30', fontweight='bold')

        n_nan_cat = sum(is_nan)
        ax.set_title(
            f'{label}  —  {len(cols) - n_nan_cat}/{len(cols)} ok'
            + (f'  ({n_nan_cat} NaN)' if n_nan_cat else ''),
            fontsize=9, fontweight='bold', pad=4
        )
        ax.set_xticks(x)
        ax.set_xticklabels([c.replace('_', '_\n') for c in cols],
                           fontsize=6, rotation=30, ha='right')
        ax.tick_params(labelsize=7)
        ax.spines[['top','right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3, linewidth=0.5)
        if ax_row == 0:
            ax.legend(fontsize=8, loc='upper right')

    fig.suptitle(
        f'Patient: {row[id_col]}  |  Group: {row["_group"]}  '
        f'|  NaN: {n_nan}/{len(num_cols)}',
        fontsize=12, fontweight='bold', y=1.005
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Patient vs group: z-score comparison ─────────────────────────────
# Shows how many standard deviations the patient sits from their group mean.

if not match.empty:
    row = match.iloc[0]
    group_df = df[df['_group'] == group]

    z_scores = {}
    for c in num_cols:
        if pd.isna(row[c]):
            continue
        g_vals = group_df[c].dropna()
        if len(g_vals) < 3:
            continue
        mu, sigma = g_vals.mean(), g_vals.std()
        if sigma > 0:
            z_scores[c] = (row[c] - mu) / sigma

    z_series = pd.Series(z_scores).sort_values(key=abs, ascending=False)
    top = z_series.head(30)  # top 30 most extreme

    fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(top) + 1.5)))
    colors = ['#D85A30' if z > 0 else '#378ADD' for z in top.values[::-1]]
    ax.barh(top.index[::-1], top.values[::-1], color=colors, edgecolor='white', height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline( 2, color='#D85A30', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(-2, color='#378ADD', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel('Z-score vs group mean', fontsize=10)
    ax.set_title(
        f'Top 30 most extreme features: {row[id_col]} vs group {group}\n'
        '(dashed lines = ±2 SD)',
        fontsize=11, pad=8
    )
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()